In [1]:
!pip install requests torch numpy --quiet

zsh:1: command not found: pip


In [2]:
import struct
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import requests

ModuleNotFoundError: No module named 'numpy'

In [ ]:
BUTTON_BITS = [
    0x8000, 0x4000, 0x2000, 0x1000,
    0x0800, 0x0400, 0x0200, 0x0100,
    0x0020, 0x0010, 0x0008, 0x0004, 0x0002, 0x0001,
]
BUTTON_NAMES = [
    "A", "B", "Z", "Start",
    "DUp", "DDown", "DLeft", "DRight",
    "L", "R", "CUp", "CDown", "CLeft", "CRight",
]

# Input dimension: X-axis, Y-axis, 14 buttons = 16 total
INPUT_DIM = 16


def parse_m64(raw: bytes) -> torch.Tensor:
    """Parse raw .m64 bytes into a (T, 16) float32 tensor.
    Columns: [x_axis, y_axis, A, B, Z, Start, DUp, DDown, DLeft, DRight, L, R, CUp, CDown, CLeft, CRight]
    """
    version = struct.unpack_from("<I", raw, 0x004)[0]
    data_start = 0x200 if version in (1, 2) else 0x400
    num_samples = struct.unpack_from("<I", raw, 0x018)[0]

    buf = np.frombuffer(
        raw[data_start : data_start + num_samples * 4], dtype=np.uint8
    ).reshape(num_samples, 4)

    y_axis = buf[:, 0].view(np.int8).astype(np.float32)   # up = positive
    x_axis = buf[:, 1].view(np.int8).astype(np.float32)   # right = positive
    btn_raw = buf[:, 2].astype(np.uint16) * 256 + buf[:, 3].astype(np.uint16)

    buttons = np.stack(
        [(btn_raw & bit).astype(bool) for bit in BUTTON_BITS], axis=1
    ).astype(np.float32)  # (T, 14)

    tensor = torch.cat([
        torch.from_numpy(x_axis).unsqueeze(1),
        torch.from_numpy(y_axis).unsqueeze(1),
        torch.from_numpy(buttons),
    ], dim=1)  # (T, 16)

    return tensor


def load_m64(source: str, cache_dir: str = ".") -> torch.Tensor:
    """Load an .m64 from a local path or URL, caching downloads locally."""
    if source.startswith("http://") or source.startswith("https://"):
        cache_path = Path(cache_dir) / Path(source).name
        if not cache_path.exists():
            print(f"Downloading {source} ...")
            cache_path.write_bytes(requests.get(source).content)
        raw = cache_path.read_bytes()
    else:
        raw = Path(source).read_bytes()
    return parse_m64(raw)

In [ ]:
class M64WindowDataset(Dataset):
    """Sliding-window dataset over one or more .m64 files.

    Each item is a (window_size, 16) float32 tensor of controller inputs.
    `file_idx` metadata is stored in self.index so you can trace which
    file and frame offset each sample comes from.

    Args:
        sources:     list of local paths or URLs to .m64 files.
        window_size: number of consecutive frames per sample.
        stride:      step between windows (default = window_size, no overlap).
        cache_dir:   directory to cache downloaded files.
    """

    def __init__(
        self,
        sources: list[str],
        window_size: int = 64,
        stride: int | None = None,
        cache_dir: str = ".",
    ):
        self.window_size = window_size
        self.stride = stride if stride is not None else window_size

        self.tensors: list[torch.Tensor] = []
        self.file_names: list[str] = []
        # index: list of (file_idx, frame_offset) for each dataset item
        self.index: list[tuple[int, int]] = []

        for file_idx, src in enumerate(sources):
            print(f"[{file_idx}] Loading: {src}")
            t = load_m64(src, cache_dir=cache_dir)
            self.tensors.append(t)
            self.file_names.append(Path(src).name)
            num_frames = t.shape[0]
            for start in range(0, num_frames - window_size + 1, self.stride):
                self.index.append((file_idx, start))

        print(f"\nTotal windows: {len(self.index):,}")

    def __len__(self) -> int:
        return len(self.index)

    def __getitem__(self, idx: int) -> dict:
        file_idx, frame_start = self.index[idx]
        window = self.tensors[file_idx][frame_start : frame_start + self.window_size]
        return {
            "inputs": window,           # (window_size, 16)
            "file_idx": file_idx,       # which .m64 file
            "file_name": self.file_names[file_idx],
            "frame_start": frame_start, # first frame index within that file
        }

In [ ]:
# --- Configure your sources here ---
# Can be local paths or raw GitHub URLs.
# Example (replace with your actual file URL or path):
SOURCES = [
    "sm64120stars__U_.m64",  # local file
    # "https://raw.githubusercontent.com/<user>/<repo>/main/sm64120stars__U_.m64",
]

WINDOW_SIZE = 64   # frames per sample
STRIDE      = 32   # 50% overlap
BATCH_SIZE  = 128

dataset = M64WindowDataset(SOURCES, window_size=WINDOW_SIZE, stride=STRIDE)

def collate_fn(batch):
    return {
        "inputs":      torch.stack([b["inputs"] for b in batch]),
        "file_idx":    torch.tensor([b["file_idx"] for b in batch]),
        "file_names":  [b["file_name"] for b in batch],
        "frame_start": torch.tensor([b["frame_start"] for b in batch]),
    }

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)

print(f"Batches per epoch: {len(loader):,}")

In [ ]:
# Inspect one batch
batch = next(iter(loader))
print("inputs shape :", batch["inputs"].shape)      # (B, window_size, 16)
print("file_idx     :", batch["file_idx"][:5])
print("file_names   :", batch["file_names"][:3])
print("frame_start  :", batch["frame_start"][:5])
print()
print("Column layout (dim=-1):")
print(f"  col 0  = x_axis")
print(f"  col 1  = y_axis")
for i, name in enumerate(BUTTON_NAMES):
    print(f"  col {i+2:<2d} = {name}")